#### Goal: Create a script to convert RDS in markdown format to JSONL dataset

In [25]:
import re
import json
import pickle
from pathlib import Path
from typing import List, Optional, Dict, Any, Literal
from sentence_transformers import SentenceTransformer

def extract_rule_info(md_content: str) -> Dict[str, Optional[Any]]:
    def find_field(patterns: List[str], content: str = md_content, flags: int = re.IGNORECASE) -> Optional[str]:
        for pat in patterns:
            match = re.search(pat, content, flags)
            if match:
                return match.group(1).strip()
        return None

    rule_id: Optional[str] = find_field([r"Rule[:\s-]*([\w-]+)"])
    schema_version: Optional[str] = find_field([r"\*\*Schema Version:\*\*\s*([\w\.-]+)", r"Schema Version:\s*([\w\.-]+)"])
    mandatory_rule: Optional[str] = find_field([r"\*\*Mandatory Rule:\*\*\s*([\w]+)", r"Mandatory Rule:\s*([\w]+)"])
    rule_description: Optional[str] = find_field([r"\*\*Rule Description:\*\*\s*([\s\S]+?)(?:\n|$)", r"Rule Description:\s*([\s\S]+?)(?:\n|$)"])
    rule_assertion: Optional[str] = find_field([r"\*\*Rule Assertion:\*\*\s*([\s\S]+?)(?:\n|$)", r"Rule Assertion:\s*([\s\S]+?)(?:\n|$)"])
    appendix_g_section: Optional[str] = find_field([r"\*\*Appendix G Section:\*\*\s*([\s\S]+?)(?:\n|$)", r"Appendix G Section:\s*([\s\S]+?)(?:\n|$)", r"Appendix G Section Reference:\s*([\s\S]+?)(?:\n|$)"])
    data_lookup: Optional[str] = find_field([r"\*\*Data Lookup:\*\*\s*([\s\S]+?)(?:\n|$)", r"Data Lookup:\s*([\s\S]+?)(?:\n|$)"])
    evaluation_context: Optional[str] = find_field([r"\*\*Evaluation Context:\*\*\s*([\s\S]+?)(?:\n|$)", r"Evaluation Context:\s*([\s\S]+?)(?:\n|$)"])
    applicability_checks: Optional[str] = find_field([r"\*\*Applicability Checks:\*\*\s*([\s\S]+?)(?:\n|$)", r"Applicability Checks:\s*([\s\S]+?)(?:\n|$)"])
    rule_logic: Optional[str] = find_field([r'(?:\*\*Rule Logic:\*\*|Rule Logic:)\s*([\s\S]+?)(?=\*\*Rule Assertion:\*\*|Rule Assertion:|$)'])
    # Special handling for rule_logic_steps due to more complex context
    def find_rule_logic_steps() -> Optional[str]:
        match = re.search(r'## Rule Logic:[\s\S]*?(?:\*\*Rule Assertion:\*\*|Rule Assertion:)\s*([\s\S]+?)(?=\*\*Notes/Questions:\*\*|Notes/Questions:|\\*\\*\[Back\]\(\.\./_toc.md\)\\*\\*|\[Back\]\(\.\./_toc.md\)|^## |\Z)', md_content, re.IGNORECASE | re.MULTILINE)
        return match.group(1).strip() if match else None

    return {
        "rule_id": rule_id,
        "schema_version": schema_version,
        "mandatory_rule": mandatory_rule,
        "rule_description": rule_description,
        "rule_assertion": rule_assertion,
        "Appendix_G_section": appendix_g_section,
        "data_lookup": data_lookup,
        "evaluation_context": evaluation_context,
        "applicability_checks": applicability_checks,
        "rule_logic": rule_logic,
        "rule_logic_steps": find_rule_logic_steps()
    }

def convert_markdown_to_jsonl(md_folders: List[str] | str, output_file: str, embed_model_name: Optional[str] = None, save_format: Literal['jsonl', 'pkl'] = 'jsonl') -> None:
    if isinstance(md_folders, str):
        md_folders = [md_folders]
    model: Optional[SentenceTransformer] = None
    if embed_model_name:
        model = SentenceTransformer(embed_model_name)
    dataset: List[Dict[str, Any]] = []
    for md_folder in md_folders:
        md_files = Path(md_folder).glob("*.md")
        for md_file in md_files:
            with open(md_file, "r", encoding="utf-8") as f:
                content = f.read()
            rule_info = extract_rule_info(content)
            # If fallback to filename, strip 'Rule' prefix if present
            if not rule_info["rule_id"]:
                stem = md_file.stem
                rule_id = re.sub(r"^Rule[:\s-]*", "", stem, flags=re.IGNORECASE)
                rule_info["rule_id"] = rule_id
            # Add embedding if model is provided and rule_description exists
            if model and rule_info.get("rule_description"):
                rule_info["embedding"] = model.encode(rule_info["rule_description"], normalize_embeddings=True).tolist()
            dataset.append(rule_info)
    if save_format == 'jsonl':
        with open(f"{output_file}.{save_format}", "w", encoding="utf-8") as out_f:
            for rule_info in dataset:
                out_f.write(json.dumps(rule_info) + "\n")
    elif save_format == 'pkl':
        with open(f"{output_file}.{save_format}", "wb") as out_f:
            pickle.dump(dataset, out_f)
    else:
        raise ValueError("save_format must be either 'jsonl' or 'pkl'")

In [26]:
SECTION_LIST = ["section1", "section4", "section5", "section6", "section10", "section11", "section12", "section16", 
                "section18", "section19", "section21", "section22", "section23"]

In [27]:
#md_folders = [f"../../../../docs/ashrae_90p1_2019/{section}" for section in SECTION_LIST]
#output_jsonl = "../data/RDS_2019.jsonl"


md_folders = [f"../../../../docs/ashrae_90p1_2019/{section}" for section in ["section22"]]
output_jsonl = "../data/RDS_2019_short_MiniLM_L6_v2"
convert_markdown_to_jsonl(md_folders, output_jsonl, "all-MiniLM-L6-v2", "pkl")

print(f"Conversion complete. Output: {output_jsonl}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Conversion complete. Output: ../data/RDS_2019_short_MiniLM_L6_v2


In [28]:
md_folders = [f"../../../../docs/ashrae_90p1_2019/{section}" for section in ["section22"]]
output_jsonl = "../data/RDS_2019_short_e5_base_v2"
convert_markdown_to_jsonl(md_folders, output_jsonl, "intfloat/e5-base-v2", "pkl")

print(f"Conversion complete. Output: {output_jsonl}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Conversion complete. Output: ../data/RDS_2019_short_e5_base_v2


In [29]:
md_folders = [f"../../../../docs/ashrae_90p1_2019/{section}" for section in ["section22"]]
output_jsonl = "../data/RDS_2019_short"
convert_markdown_to_jsonl(md_folders, output_jsonl, None, "jsonl")

print(f"Conversion complete. Output: {output_jsonl}")

Conversion complete. Output: ../data/RDS_2019_short
